**Phase 3: Uploading Dataset from Google Drive to Google Collab**

Step 1: Mount Drive

In [46]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Step: Copy dataset to LOCAL

In [47]:
!cp -r "/content/drive/MyDrive/#08 MSc Data Science/7FTC2001 Data Science Project/emotion_dataset" /content/

Step: Set Local Path

In [48]:
DATASET_PATH = '/content/emotion_dataset'

**PHASE 5 — TRAIN MODEL (GPU)**

Step 1: Install Libraries

In [ ]:
!pip install tensorflow opencv-python

Step 2: Image Data Generator

In [49]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=5,
    zoom_range=0.02,
    horizontal_flip=True,
    brightness_range=[0.8,1.2]   # 🔥 NEW (important)
)

Step : Dataset for Training

In [50]:
train_gen = datagen.flow_from_directory(
    DATASET_PATH + '/train',
    target_size=(64,64),
    batch_size=64,
    class_mode='categorical',
    color_mode='grayscale'
)

Found 2808 images belonging to 5 classes.


Step: Dataset for Validation

In [51]:
val_gen = datagen.flow_from_directory(
    DATASET_PATH + '/val',
    target_size=(64,64),
    batch_size=64,
    class_mode='categorical',
    color_mode='grayscale',
    shuffle=False
)

Found 1430 images belonging to 5 classes.


Step  : PRINT LABEL ORDER

In [52]:
print(train_gen.class_indices)

{'angry': 0, 'happy': 1, 'neutral': 2, 'sad': 3, 'surprise': 4}


Step 4: Build Model

In [53]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(64,64,1)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(5, activation='softmax')
])

Step 5: Compile Model

In [54]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

Step : Add Early Stopping

In [55]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

Step 6: Train Model

In [56]:
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=[early_stop]
)

Epoch 1/15
44/44 ━━━━━━━━━━━━━━━━━━━━ 21s 410ms/step - accuracy: 0.2101 - loss: 1.6116 - val_accuracy: 0.3280 - val_loss: 1.6054
Epoch 2/15
44/44 ━━━━━━━━━━━━━━━━━━━━ 12s 283ms/step - accuracy: 0.2432 - loss: 1.6014 - val_accuracy: 0.3832 - val_loss: 1.5547
Epoch 3/15
44/44 ━━━━━━━━━━━━━━━━━━━━ 13s 288ms/step - accuracy: 0.3109 - loss: 1.5417 - val_accuracy: 0.4231 - val_loss: 1.3128
Epoch 4/15
44/44 ━━━━━━━━━━━━━━━━━━━━ 12s 280ms/step - accuracy: 0.3486 - loss: 1.4645 - val_accuracy: 0.4720 - val_loss: 1.2166
Epoch 5/15
44/44 ━━━━━━━━━━━━━━━━━━━━ 12s 282ms/step - accuracy: 0.4330 - loss: 1.3496 - val_accuracy: 0.4741 - val_loss: 1.1561
Epoch 6/15
44/44 ━━━━━━━━━━━━━━━━━━━━ 12s 282ms/step - accuracy: 0.4957 - loss: 1.2418 - val_accuracy: 0.5601 - val_loss: 1.0653
Epoch 7/15
44/44 ━━━━━━━━━━━━━━━━━━━━ 13s 289ms/step - accuracy: 0.5477 - loss: 1.1133 - val_accuracy: 0.5825 - val_loss: 1.0021
Epoch 8/15
44/44 ━━━━━━━━━━━━━━━━━━━━ 12s 278ms/step - accuracy: 0.6108 - loss: 0.9853 - val_accu

Step: Model Evaluation

In [57]:
val_loss, val_acc = model.evaluate(val_gen)
print("Validation Accuracy:", val_acc)

23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 180ms/step - accuracy: 0.7441 - loss: 0.7118
Validation Accuracy: 0.7440559267997742


**PHASE 6 — SAVE MODEL**

Step 1: Save Model

In [58]:
model.save('emotion_model.keras')

Step 2: Download Model

In [59]:
from google.colab import files
files.download('emotion_model.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Check Keras version

In [ ]:
import tensorflow as tf
import keras

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)

TensorFlow version: 2.19.0
Keras version: 3.13.2
